# WM-811K Wafer Map — Training Pipeline

Load the dataset, filter out unknown failure types, stratified 60/20/20 split, and build PyTorch `Dataset` / `DataLoader`.

In [1]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from scipy.ndimage import zoom
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

## 1. Load the data

In [2]:
import pickle
import sys

# Patch old pandas module paths for compatibility
# The pickle was saved with pandas < 0.20 which used pandas.indexes.*
_compat_map = {
    'pandas.indexes':         'pandas.core.indexes',
    'pandas.indexes.base':    'pandas.core.indexes.base',
    'pandas.indexes.numeric': 'pandas.core.indexes.base',
    'pandas.indexes.range':   'pandas.core.indexes.range_',
    'pandas.indexes.multi':   'pandas.core.indexes.multi',
    'pandas.indexes.frozen':  'pandas.core.indexes.frozen',
}

for old, new in _compat_map.items():
    try:
        sys.modules[old] = __import__(new, fromlist=[''])
    except ImportError:
        pass

class _CompatUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        for old, new in _compat_map.items():
            if module.startswith(old):
                module = module.replace(old, new, 1)
                break
        return super().find_class(module, name)

with open("./Data/LSWMD.pkl", "rb") as f:
    try:
        df = pickle.load(f, encoding='latin1')
    except (ModuleNotFoundError, ImportError):
        f.seek(0)
        df = _CompatUnpickler(f)
        df.encoding = 'latin1'
        df = df.load()

print(f"Shape: {df.shape}")
df.head()

/var/folders/k2/4b0bgh394psfjhjl_xtj93gh000585/T/ipykernel_41093/2994757470.py:31: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  df = pickle.load(f, encoding='latin1')


Shape: (811457, 6)


,waferMap,dieSize,lotName,waferIndex,trianTestLabel,failureType
0,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",1683.0,lot1,1.0,[[Training]],[[none]]
1,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",1683.0,lot1,2.0,[[Training]],[[none]]
2,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",1683.0,lot1,3.0,[[Training]],[[none]]
3,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",1683.0,lot1,4.0,[[Training]],[[none]]
4,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",1683.0,lot1,5.0,[[Training]],[[none]]


## 2. Extract labels and filter out unknown failure types

The `failureType` column is stored as a nested array. Rows with an empty array (no label) or `'unknown'` are dropped — keeping only the 9 named defect classes plus `none`.

In [3]:
def extract_label(x):
    if isinstance(x, (list, np.ndarray)):
        flat = np.array(x).flatten()
        if len(flat) > 0:
            return str(flat[0])
    if isinstance(x, str):
        return x
    return "unknown"

df['failure_label'] = df['failureType'].apply(extract_label)

# Drop unlabeled rows
df_labeled = df[~df['failure_label'].isin(['unknown', ''])].reset_index(drop=True)

print(f"Before filter: {len(df):,}")
print(f"After filter:  {len(df_labeled):,}")
print(f"\nClass counts:")
print(df_labeled['failure_label'].value_counts())

Before filter: 811,457
After filter:  172,950

Class counts:
failure_label
none         147431
Edge-Ring      9680
Edge-Loc       5189
Center         4294
Loc            3593
Scratch        1193
Random          866
Donut           555
Near-full       149
Name: count, dtype: int64


## 3. Stratified 60 / 20 / 20 split

Each class keeps the same proportion in train, validation, and test.

In [4]:
classes = sorted(df_labeled['failure_label'].unique().tolist())
class_to_idx = {c: i for i, c in enumerate(classes)}
idx_to_class = {i: c for c, i in class_to_idx.items()}
NUM_CLASSES = len(classes)
print(f"{NUM_CLASSES} classes: {classes}")

df_labeled['label_idx'] = df_labeled['failure_label'].map(class_to_idx)

# 60/20/20 stratified: first carve off 20% test, then split the remaining 80% into 60/20 (i.e. 0.25 of 0.8).
y = df_labeled['label_idx'].values
idx_all = np.arange(len(df_labeled))

idx_trainval, idx_test = train_test_split(
    idx_all, test_size=0.20, stratify=y, random_state=SEED
)
idx_train, idx_val = train_test_split(
    idx_trainval, test_size=0.25, stratify=y[idx_trainval], random_state=SEED
)

print(f"\nTrain: {len(idx_train):,}  ({len(idx_train)/len(idx_all):.1%})")
print(f"Val:   {len(idx_val):,}  ({len(idx_val)/len(idx_all):.1%})")
print(f"Test:  {len(idx_test):,}  ({len(idx_test)/len(idx_all):.1%})")

# Verify class proportions match across splits
prop_table = pd.DataFrame({
    'all':   df_labeled['failure_label'].value_counts(normalize=True),
    'train': df_labeled.iloc[idx_train]['failure_label'].value_counts(normalize=True),
    'val':   df_labeled.iloc[idx_val]['failure_label'].value_counts(normalize=True),
    'test':  df_labeled.iloc[idx_test]['failure_label'].value_counts(normalize=True),
}).fillna(0).round(4)
print("\nClass proportions per split:")
prop_table

9 classes: ['Center', 'Donut', 'Edge-Loc', 'Edge-Ring', 'Loc', 'Near-full', 'Random', 'Scratch', 'none']

Train: 103,770  (60.0%)
Val:   34,590  (20.0%)
Test:  34,590  (20.0%)

Class proportions per split:


,all,train,val,test
failure_label,,,,
none,0.8524,0.8525,0.8524,0.8524
Edge-Ring,0.0560,0.0560,0.0560,0.0560
Edge-Loc,0.0300,0.0300,0.0300,0.0300
Center,0.0248,0.0248,0.0248,0.0248
Loc,0.0208,0.0208,0.0208,0.0208
Scratch,0.0069,0.0069,0.0069,0.0069
Random,0.0050,0.0050,0.0050,0.0050
Donut,0.0032,0.0032,0.0032,0.0032
Near-full,0.0009,0.0009,0.0009,0.0009


## 4. PyTorch `Dataset`

Wafer maps come in many sizes (heights/widths range widely) so each map is resized to a fixed `IMG_SIZE × IMG_SIZE` with nearest-neighbor (preserves the discrete 0/1/2 codes). The result is returned as a `(1, H, W)` float tensor in `[0, 1]` plus the integer class label.

Training instances also get random horizontal + vertical flips (each with p = 0.5). Wafer-map defects don't have a canonical orientation, so flipping is label-preserving — it effectively quadruples the variety the model sees per class, which is especially useful for the rare classes the weighted sampler keeps drawing. Validation and test stay deterministic.

In [5]:
IMG_SIZE = 64

class WaferMapDataset(Dataset):
    def __init__(self, df, indices, img_size=IMG_SIZE, augment: bool = False):
        self.maps    = df['waferMap'].values[indices]
        self.labels  = df['label_idx'].values[indices].astype(np.int64)
        self.img_size = img_size
        self.augment = augment

    def __len__(self):
        return len(self.labels)

    def _resize(self, wmap):
        wmap = np.asarray(wmap)
        if wmap.ndim < 2 or wmap.shape[0] == 0 or wmap.shape[1] == 0:
            return np.zeros((self.img_size, self.img_size), dtype=np.float32)
        h, w = wmap.shape
        return zoom(wmap, (self.img_size / h, self.img_size / w), order=0)

    def __getitem__(self, i):
        wmap = self._resize(self.maps[i]).astype(np.float32) / 2.0  # 0/1/2 -> [0, 0.5, 1]
        if self.augment:
            # Independent p=0.5 horizontal and vertical flips. Wafer defects have no
            # canonical orientation so flips are label-preserving.
            if np.random.rand() < 0.5:
                wmap = np.ascontiguousarray(wmap[:, ::-1])
            if np.random.rand() < 0.5:
                wmap = np.ascontiguousarray(wmap[::-1, :])
        x = torch.from_numpy(wmap).unsqueeze(0)                     # (1, H, W)
        y = torch.tensor(self.labels[i], dtype=torch.long)
        return x, y

train_ds = WaferMapDataset(df_labeled, idx_train, augment=True)
val_ds   = WaferMapDataset(df_labeled, idx_val,   augment=False)
test_ds  = WaferMapDataset(df_labeled, idx_test,  augment=False)

print(f"train_ds: {len(train_ds):,}  (augmented)")
print(f"val_ds:   {len(val_ds):,}")
print(f"test_ds:  {len(test_ds):,}")

x0, y0 = train_ds[0]
print(f"\nsample x: shape={tuple(x0.shape)}, dtype={x0.dtype}, min={x0.min():.2f}, max={x0.max():.2f}")
print(f"sample y: {y0.item()} ({idx_to_class[y0.item()]})")

train_ds: 103,770  (augmented)
val_ds:   34,590
test_ds:  34,590

sample x: shape=(1, 64, 64), dtype=torch.float32, min=0.00, max=1.00
sample y: 8 (none)


## 5. `DataLoader`s

In [6]:
from torch.utils.data import WeightedRandomSampler

BATCH_SIZE  = 64
NUM_WORKERS = 0

# Per-sample weights: 1 / count[class]. Rare classes get higher draw probability,
# so each batch ends up with ~uniform class composition. Replacement=True is required
# (otherwise rare classes run out and the distribution skews back toward the majority).
train_labels = df_labeled['label_idx'].values[idx_train]
class_counts = np.bincount(train_labels, minlength=NUM_CLASSES)
class_weights = 1.0 / class_counts
sample_weights = class_weights[train_labels]

train_sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(train_labels),  # one "epoch" = same #steps as natural dataset
    replacement=True,
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=train_sampler,
                          num_workers=NUM_WORKERS, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS)

# --- class counts BEFORE / AFTER augmentation ---
# "Before" = raw training set (pre-sampler, no flips).
# "After"  = one epoch as the model actually sees it: WeightedRandomSampler drawing
#            with replacement, where each draw also gets an independent random
#            hflip+vflip from WaferMapDataset(augment=True).
#            We don't load images here — we just iterate sampler indices and look up
#            their labels. Flipping doesn't change a sample's class, so this is the
#            true post-augmentation class distribution.
g = torch.Generator().manual_seed(SEED)
sim_sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(train_labels),
    replacement=True,
    generator=g,
)
sampled_idx = np.fromiter(iter(sim_sampler), dtype=np.int64, count=len(train_labels))
after_counts = np.bincount(train_labels[sampled_idx], minlength=NUM_CLASSES)

print(f"{'class':12s} {'before':>10s} {'after':>10s} {'before %':>10s} {'after %':>10s}")
print('-' * 56)
tot_b, tot_a = class_counts.sum(), after_counts.sum()
for i in range(NUM_CLASSES):
    print(f"{idx_to_class[i]:12s} "
          f"{class_counts[i]:>10,d} {after_counts[i]:>10,d} "
          f"{class_counts[i]/tot_b:>9.2%} {after_counts[i]/tot_a:>9.2%}")
print('-' * 56)
print(f"{'total':12s} {tot_b:>10,d} {tot_a:>10,d}")

# DataLoader sanity check on one batch
xb, yb = next(iter(train_loader))
print(f"\nbatch x: {tuple(xb.shape)}  dtype={xb.dtype}")
print(f"batch y: {tuple(yb.shape)}  dtype={yb.dtype}")

class            before      after   before %    after %
--------------------------------------------------------
Center            2,576     11,597     2.48%    11.18%
Donut               333     11,605     0.32%    11.18%
Edge-Loc          3,113     11,414     3.00%    11.00%
Edge-Ring         5,808     11,531     5.60%    11.11%
Loc               2,156     11,419     2.08%    11.00%
Near-full            89     11,487     0.09%    11.07%
Random              520     11,493     0.50%    11.08%
Scratch             716     11,511     0.69%    11.09%
none             88,459     11,713    85.25%    11.29%
--------------------------------------------------------
total           103,770    103,770

batch x: (64, 1, 64, 64)  dtype=torch.float32
batch y: (64,)  dtype=torch.int64


## 6. Model 1 — Simple CNN

In [19]:
import torch
import torch.nn as nn


class Model1(nn.Module):
    """Simple CNN with BatchNorm to fight overfitting.

    Normalization additions vs. the original:
      - BatchNorm2d after each conv (before ReLU). Stabilizes activations and
        also acts as a regularizer because batch statistics inject noise.
      - BatchNorm1d after the first FC layer.
      - Light spatial dropout (Dropout2d) in the conv stack on top of the existing
        FC dropout, so regularization isn't only at the very end.
    """
    def __init__(self, num_classes: int = 9, in_channels: int = 1):
        super().__init__()

        # Convolutional part
        self.conv1 = nn.Conv2d(in_channels, 32, kernel_size=6, padding=1)
        self.bn1   = nn.BatchNorm2d(32)
        self.relu1 = nn.ReLU(inplace=True)
        self.pool1 = nn.MaxPool2d(2)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=6, padding=1)
        self.bn2   = nn.BatchNorm2d(64)
        self.relu2 = nn.ReLU(inplace=True)
        self.pool2 = nn.MaxPool2d(2)

        self.conv3 = nn.Conv2d(64, 128, kernel_size=6, padding=1)
        self.bn3   = nn.BatchNorm2d(128)
        self.relu3 = nn.ReLU(inplace=True)
        self.pool3 = nn.MaxPool2d(2)

        self.conv_dropout = nn.Dropout2d(p=0.1)

        # Dry-run to infer flattened size
        with torch.no_grad():
            dummy = torch.zeros(1, in_channels, 64, 64)
            dummy = self.pool3(self.relu3(self.bn3(self.conv3(
                     self.pool2(self.relu2(self.bn2(self.conv2(
                     self.pool1(self.relu1(self.bn1(self.conv1(dummy)))))))))))) 
            flat_size = dummy.numel()

        # Fully connected part
        self.flatten = nn.Flatten()
        self.fc1     = nn.Linear(flat_size, 256)
        self.bn_fc1  = nn.BatchNorm1d(256)
        self.relu4   = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(p=0.5)
        self.fc2     = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.pool1(self.relu1(self.bn1(self.conv1(x))))
        x = self.pool2(self.relu2(self.bn2(self.conv2(x))))
        x = self.pool3(self.relu3(self.bn3(self.conv3(x))))
        x = self.conv_dropout(x)

        x = self.flatten(x)
        x = self.relu4(self.bn_fc1(self.fc1(x)))
        x = self.dropout(x)
        x = self.fc2(x)
        return x


device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print(f"device: {device}")

# Param count via a throwaway instance — does NOT bind `model1`, so re-running this
# cell never resets a trained model. The actual `model1` is built in the training cell.
_probe = Model1(num_classes=NUM_CLASSES)
print(f"Model1 parameters: {sum(p.numel() for p in _probe.parameters()):,}")
del _probe

device: mps
Model1 parameters: 1,192,745


## 7. Train Model 1

In [20]:
import time

EPOCHS = 50
LR     = 1e-3

# Build the model + optimizer here, alongside the training loop. This couples them
# so that re-running the model-definition cell (cell 6) cannot silently wipe a trained
# `model1` and leave the test-eval cell evaluating random weights. To retrain from
# scratch, re-run *this* cell.
model1 = Model1(num_classes=NUM_CLASSES).to(device)

# Forward-pass sanity check
with torch.no_grad():
    out = model1(xb.to(device))
print(f"forward output: {tuple(out.shape)}  (expected ({BATCH_SIZE}, {NUM_CLASSES}))")

optimizer = torch.optim.Adam(model1.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()


def run_epoch(model, loader, train: bool):
    model.train(train)
    total_loss = 0.0
    total_correct = 0
    total_seen = 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss    += loss.item() * yb.size(0)
            total_correct += (logits.argmax(1) == yb).sum().item()
            total_seen    += yb.size(0)
    return total_loss / total_seen, total_correct / total_seen


history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = run_epoch(model1, train_loader, train=True)
    va_loss, va_acc = run_epoch(model1, val_loader,   train=False)
    history['train_loss'].append(tr_loss); history['train_acc'].append(tr_acc)
    history['val_loss'].append(va_loss);   history['val_acc'].append(va_acc)
    print(f"epoch {epoch:2d} | "
          f"train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
          f"val loss {va_loss:.4f} acc {va_acc:.4f} | "
          f"{time.time()-t0:.1f}s")


forward output: (64, 9)  (expected (64, 9))
epoch  1 | train loss 0.4788 acc 0.8219 | val loss 1.1537 acc 0.5701 | 36.0s
epoch  2 | train loss 0.2748 acc 0.9014 | val loss 0.1629 acc 0.9502 | 35.5s
epoch  3 | train loss 0.2002 acc 0.9297 | val loss 0.1386 acc 0.9561 | 35.6s
epoch  4 | train loss 0.1594 acc 0.9440 | val loss 0.3034 acc 0.9062 | 35.5s
epoch  5 | train loss 0.1324 acc 0.9538 | val loss 0.1441 acc 0.9545 | 35.5s
epoch  6 | train loss 0.1114 acc 0.9616 | val loss 0.1516 acc 0.9544 | 35.6s
epoch  7 | train loss 0.0953 acc 0.9677 | val loss 0.1988 acc 0.9375 | 35.5s
epoch  8 | train loss 0.0840 acc 0.9715 | val loss 0.1118 acc 0.9664 | 35.5s
epoch  9 | train loss 0.0756 acc 0.9744 | val loss 0.3198 acc 0.9050 | 35.5s
epoch 10 | train loss 0.0712 acc 0.9762 | val loss 0.2078 acc 0.9353 | 35.5s
epoch 11 | train loss 0.0636 acc 0.9781 | val loss 0.3111 acc 0.9022 | 35.6s
epoch 12 | train loss 0.0596 acc 0.9798 | val loss 0.1377 acc 0.9596 | 35.5s
epoch 13 | train loss 0.0544 acc

## 8. Evaluate Model 1 on the test set

In [21]:
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score, cohen_kappa_score,
)


@torch.no_grad()
def collect_predictions(model, loader):
    model.eval()
    ys, ps = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        ps.append(model(xb).argmax(1).cpu().numpy())
        ys.append(yb.numpy())
    return np.concatenate(ys), np.concatenate(ps)


y_true, y_pred = collect_predictions(model1, test_loader)
target_names = [idx_to_class[i] for i in range(NUM_CLASSES)]

print(f"Test samples: {len(y_true):,}")
print(f"\nAccuracy:           {accuracy_score(y_true, y_pred):.4f}")
print(f"Balanced accuracy:  {balanced_accuracy_score(y_true, y_pred):.4f}  (mean per-class recall)")
print(f"Cohen's kappa:      {cohen_kappa_score(y_true, y_pred):.4f}")
for avg in ('macro', 'weighted'):
    p = precision_score(y_true, y_pred, average=avg, zero_division=0)
    r = recall_score(   y_true, y_pred, average=avg, zero_division=0)
    f = f1_score(       y_true, y_pred, average=avg, zero_division=0)
    print(f"{avg:>8s}: precision {p:.4f}  recall {r:.4f}  f1 {f:.4f}")

print("\nPer-class report:")
print(classification_report(y_true, y_pred, target_names=target_names,
                            digits=4, zero_division=0))

# Confusion matrix as a labeled DataFrame (rows = true, columns = predicted)
cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
cm_df = pd.DataFrame(cm, index=target_names, columns=target_names)
print("Confusion matrix (rows=true, cols=predicted):")
cm_df

Test samples: 34,590

Accuracy:           0.9692
Balanced accuracy:  0.8608  (mean per-class recall)
Cohen's kappa:      0.8870
   macro: precision 0.8664  recall 0.8608  f1 0.8623
weighted: precision 0.9700  recall 0.9692  f1 0.9694

Per-class report:
              precision    recall  f1-score   support

      Center     0.8478    0.9534    0.8975       859
       Donut     0.8922    0.8198    0.8545       111
    Edge-Loc     0.7625    0.8102    0.7856      1038
   Edge-Ring     0.9569    0.9757    0.9662      1936
         Loc     0.7382    0.7423    0.7403       718
   Near-full     0.9310    0.9000    0.9153        30
      Random     0.8693    0.8844    0.8768       173
     Scratch     0.8100    0.6778    0.7380       239
        none     0.9895    0.9839    0.9867     29486

    accuracy                         0.9692     34590
   macro avg     0.8664    0.8608    0.8623     34590
weighted avg     0.9700    0.9692    0.9694     34590

Confusion matrix (rows=true, cols=predicte

,Center,Donut,Edge-Loc,Edge-Ring,Loc,Near-full,Random,Scratch,none
Center,819,3,0,0,10,0,2,0,25
Donut,2,91,2,1,8,0,2,0,5
Edge-Loc,5,2,841,27,41,1,4,3,114
Edge-Ring,1,0,30,1889,1,0,1,0,14
Loc,22,4,45,0,533,0,2,12,100
Near-full,1,0,1,0,0,27,1,0,0
Random,1,1,3,1,9,1,153,0,4
Scratch,0,1,7,0,23,0,1,162,45
none,115,0,174,56,97,0,10,23,29011
